# Lab 03: API Testing - Introduction

**Lab**: 03-api-testing  
**Duration**: ~5 minutes  
**Prerequisites**: Stripe account with sandbox access

## Learning Objectives

By the end of this notebook, you will:
- Understand Stripe sandboxes and their benefits
- Master the `expand` parameter for efficient API calls
- Use `metadata` to store custom data on Stripe objects
- Know the testing tools available in Stripe

---

## Why Testing Matters

Testing your Stripe integration is critical before going live. Stripe provides sandboxes that simulate real transactions without:

- Moving real money
- Charging real credit cards
- Affecting your live account data

### The Cost of Not Testing

| Scenario | Risk |
|----------|------|
| Untested payment flow | Customers can't complete purchases |
| No error handling | Poor user experience on declines |
| Subscription bugs | Incorrect billing, lost revenue |
| Missing webhooks | Out-of-sync data, manual fixes |

<!-- PRESENTER: Ask attendees about their current testing practices -->

## Stripe Sandboxes

[Sandboxes](https://docs.stripe.com/sandboxes) are isolated test environments that let you safely develop and test your Stripe integration.

### Key Benefits

| Feature | Benefit |
|---------|----------|
| Isolated environments | Up to 5 separate sandboxes |
| Granular access control | Control who can access each sandbox |
| Fully isolated settings | Sandbox settings don't affect live mode |
| Team collaboration | Dedicated environments per team |
| V1 and V2 API support | Test latest API features |

### When to Use Sandboxes

- Development and testing of new features
- CI/CD pipeline integration testing
- Multiple teams needing separate test data
- External partner testing with controlled access
- Testing V2 API features

## API Key Conventions

Stripe uses key prefixes to distinguish between environments:

| Prefix | Type | Environment |
|--------|------|-------------|
| `sk_test_` | Secret key | Sandbox |
| `pk_test_` | Publishable key | Sandbox |
| `sk_live_` | Secret key | Live mode |
| `pk_live_` | Publishable key | Live mode |

### Critical Rule

**NEVER use live keys (`sk_live_`, `pk_live_`) for testing!**

The [Stripe Services Agreement](https://stripe.com/legal/ssa) prohibits testing with real payment methods in live mode.

```python
# Safe: Sandbox key
stripe.api_key = "sk_test_..."

# DANGEROUS: Live key for testing
stripe.api_key = "sk_live_..."  # Never do this!
```

## Setup

Let's verify our environment is ready.

<!-- PRESENTER: Verify attendees have prerequisites installed -->

In [ ]:
# Environment setup and verification
import os
import stripe
from dotenv import load_dotenv


load_dotenv()
stripe.api_key = os.environ.get('STRIPE_SECRET_KEY')

# Verify we're using a sandbox key
if stripe.api_key:
    if 'test' in stripe.api_key:
        print("Safe: Using SANDBOX API key")
    else:
        print("WARNING: Using LIVE mode API key!")
        print("Please switch to a sandbox key for this workshop.")
    
    # Verify connection
    try:
        account = stripe.Account.retrieve()
        print(f"Connected to account: {account.id}")
    except stripe.error.AuthenticationError:
        print("ERROR: Invalid API key")
else:
    print("ERROR: STRIPE_API_KEY not set in .env file")

## The `expand` Parameter

By default, Stripe API responses include IDs for related objects instead of the full objects. The `expand` parameter lets you retrieve related objects in a single API call.

### Why Use Expand?

- **Reduce API calls**: Get related data in one request
- **Improve performance**: Fewer round-trips to Stripe
- **Cleaner code**: No need for multiple retrieve calls

### Example: Without vs With Expand

```python
# WITHOUT expand - payment_method is just an ID string
pi = stripe.PaymentIntent.retrieve("pi_xxx")
print(pi.payment_method)  # "pm_xxx" (just the ID)

# Need another API call to get payment method details
pm = stripe.PaymentMethod.retrieve(pi.payment_method)

# WITH expand - payment_method is the full object
pi = stripe.PaymentIntent.retrieve("pi_xxx", expand=["payment_method"])
print(pi.payment_method.card.brand)  # "visa" (full object available!)
```

In [ ]:
# Let's see expand in action
# First, create a PaymentIntent
payment_intent = stripe.PaymentIntent.create(
    amount=1000,
    currency="usd",
    payment_method="pm_card_visa",
    confirm=True,
    automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
)

print("WITHOUT expand:")
print(f"  payment_method: {payment_intent.payment_method}")
print(f"  Type: {type(payment_intent.payment_method).__name__}")

In [ ]:
# Now retrieve the same PaymentIntent WITH expand
pi_expanded = stripe.PaymentIntent.retrieve(
    payment_intent.id,
    expand=["payment_method", "latest_charge"]
)

print("WITH expand:")
print(f"  payment_method type: {type(pi_expanded.payment_method).__name__}")
print(f"  Card brand: {pi_expanded.payment_method.card.brand}")
print(f"  Card last4: {pi_expanded.payment_method.card.last4}")
print(f"  Charge ID: {pi_expanded.latest_charge.id}")
print(f"  Charge amount: ${pi_expanded.latest_charge.amount / 100:.2f}")

### Checkpoint

You should see:
- Without expand: `payment_method` is a string ID
- With expand: `payment_method` is a full object with card details

### Common Expandable Fields

| Object | Expandable Fields |
|--------|-------------------|
| PaymentIntent | `payment_method`, `customer`, `latest_charge`, `invoice` |
| Subscription | `customer`, `default_payment_method`, `latest_invoice` |
| Invoice | `customer`, `subscription`, `charge`, `payment_intent` |
| Charge | `customer`, `payment_intent`, `balance_transaction` |

## The `metadata` Field

Metadata lets you store custom key-value pairs on most Stripe objects. This is incredibly useful for:

- Linking Stripe objects to your internal systems
- Storing business context (order IDs, user IDs, campaign sources)
- Adding custom attributes without a separate database

### Metadata Rules

- Up to **50 keys** per object
- Key names: max **40 characters**
- Values: max **500 characters**
- All values are stored as **strings**

In [ ]:
# Create a customer with metadata
customer = stripe.Customer.create(
    email="workshop-demo@example.com",
    name="Workshop Demo User",
    metadata={
        "internal_user_id": "usr_12345",
        "plan_tier": "enterprise",
        "signup_source": "workshop_demo",
        "account_manager": "jane.doe"
    }
)

print(f"Customer ID: {customer.id}")
print(f"Email: {customer.email}")
print("\nMetadata:")
for key, value in customer.metadata.items():
    print(f"  {key}: {value}")

In [ ]:
# Create a PaymentIntent with metadata for order tracking
order_payment = stripe.PaymentIntent.create(
    amount=4999,  # $49.99
    currency="usd",
    customer=customer.id,
    payment_method="pm_card_visa",
    confirm=True,
    automatic_payment_methods={"enabled": True, "allow_redirects": "never"},
    metadata={
        "order_id": "ORD-2024-001234",
        "product_sku": "WIDGET-PRO-001",
        "shipping_method": "express",
        "promo_code": "WORKSHOP20"
    }
)

print(f"Payment ID: {order_payment.id}")
print(f"Amount: ${order_payment.amount / 100:.2f}")
print(f"Status: {order_payment.status}")
print("\nOrder Metadata:")
for key, value in order_payment.metadata.items():
    print(f"  {key}: {value}")

In [ ]:
# Update metadata on an existing object
updated_customer = stripe.Customer.modify(
    customer.id,
    metadata={
        "plan_tier": "premium",  # Update existing key
        "upgraded_at": "2026-01-15",  # Add new key
        "signup_source": ""  # Empty string removes the key
    }
)

print("Updated Metadata:")
for key, value in updated_customer.metadata.items():
    print(f"  {key}: {value}")

### Checkpoint

You should see:
- Customer created with 4 metadata fields
- PaymentIntent with order tracking metadata
- Updated customer with modified metadata (signup_source removed)

**Dashboard**: Navigate to [Customers](https://dashboard.stripe.com/test/customers) and click on a customer to see their metadata.

### Metadata Best Practices

1. **Use consistent key names** across your application
2. **Prefix keys** by domain (e.g., `order_`, `user_`, `campaign_`)
3. **Don't store sensitive data** (PII, passwords, secrets)
4. **Use for cross-referencing** with your internal systems

## Stripe Testing Tools Overview

This lab covers three main testing tools:

### 1. Sandboxes
Isolated environments for your team.

### 2. Test Cards
Special card numbers that simulate different payment scenarios including:
- Successful payments
- Declined payments
- Specific error conditions

### 3. Test Clocks
Simulate the passage of time for subscriptions:
- Test trial periods without waiting
- Simulate annual renewals
- Test payment failures on renewal

```
+------------------+     +------------------+     +------------------+
|    Sandboxes     |     |   Test Cards     |     |   Test Clocks    |
|                  |     |                  |     |                  |
|  Isolated envs   |     |  Payment sims    |     |  Time simulation |
|  Team access     |     |  Error handling  |     |  Billing cycles  |
+------------------+     +------------------+     +------------------+
```

## Summary

In this introduction, you learned:

- **Sandboxes** are isolated environments for safe testing
- **Sandbox API keys** (prefixed with `sk_test_`) must always be used for testing
- **`expand`** retrieves related objects in a single API call
- **`metadata`** stores custom key-value data on Stripe objects
- **Three testing tools** are available: Sandboxes, Test Cards, and Test Clocks

## Next Steps

Continue to `02_sandbox_and_test_cards.ipynb` to simulate payments with test cards!